In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip uninstall tensorflow keras segmentation-models -y


Found existing installation: tensorflow 2.15.0
Uninstalling tensorflow-2.15.0:
ERROR: Operation cancelled by user


In [ ]:
!pip install tensorflow==2.8.0 keras==2.8.0 segmentation-models


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 497.6/497.6 MB 36.8 MB/s eta 0:00:01

In [ ]:
import segmentation_models as sm

sm.set_framework('tf.keras')

sm.framework()
BACKBONE = 'resnet34'
preprocess_input = sm.get_preprocessing(BACKBONE)

Size_x = 256
Size_y = 256



In [ ]:
!pip install albumentations

In [ ]:
import albumentations as A
images_to_generate = 2000

path_x = '/kaggle/input/coalsegmentation/Criepi2009_1_Tif_CCSEM_150x/images'
path_y = '/kaggle/input/coalsegmentation/Criepi2009_1_Tif_CCSEM_150x/masks'

import os

# Define the new directory path
new_image_path = '/content/augimg/'
new_mask_path = '/content/augmask/'

# Create the new directory
os.makedirs(new_image_path, exist_ok=True)
os.makedirs(new_mask_path, exist_ok=True)

print(f"Directory '{new_image_path}' created successfully!")
print(f"Directory '{new_mask_path}' created successfully!")





In [ ]:
images = []
masks = []

# Get sorted lists of files in each directory
image_files = sorted(os.listdir(path_x))
mask_files = sorted(os.listdir(path_y))

images = sorted([os.path.join(path_x, f) for f in os.listdir(path_x) if f.endswith('.png') or f.endswith('.bmp')])
masks = sorted([os.path.join(path_y, f) for f in os.listdir(path_y) if f.endswith('.png') or f.endswith('.bmp')])




In [ ]:
aug = A.Compose([
    A.VerticalFlip(p = 0.5),
    A.RandomRotate90(p = 0.5),
    A.HorizontalFlip(p = 1),
    A.Transpose(p = 1),
    A.GridDistortion(p = 1)
])

In [ ]:
import random
import os
import imageio as io

for i in range(images_to_generate):
    number = random.randint(0, len(images) - 1)
    image_path = images[number]
    mask_path = masks[number]

    original_image = io.imread(image_path)
    original_mask = io.imread(mask_path)

    augmented = aug(image=original_image, mask=original_mask)
    transformed_image = augmented['image']
    transformed_mask = augmented['mask']

    base_filename = f"augmented_{i:04d}"
    new_img_path = os.path.join(new_image_path, f"{base_filename}.png")
    new_msk_path = os.path.join(new_mask_path, f"{base_filename}.png")

    io.imsave(new_img_path, transformed_image)
    io.imsave(new_msk_path, transformed_mask)

print("Augmentation and saving completed.")







In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from skimage.io import imread
from skimage.color import rgb2gray
from skimage.transform import resize

img_height = 256
img_width = 256

def is_image_file(filename):
    extensions = ['.jpg', '.jpeg', '.png', '.bmp']
    return any(filename.lower().endswith(ext) for ext in extensions)

Train_path_x = '/content/augimg/'
Train_path_y = '/content/augmask/'

train_files_x = sorted([f for f in os.listdir(Train_path_x) if is_image_file(f)])
train_files_y = sorted([f for f in os.listdir(Train_path_y) if is_image_file(f)])

assert len(train_files_x) == len(train_files_y), "Mismatch in number of images and masks"

X_train = np.zeros((len(train_files_x), img_width, img_height, 3), dtype=np.float32)
Y_train = np.zeros((len(train_files_y), img_width, img_height, 1), dtype=np.float32)

# Load X_train images
for n, filename in enumerate(train_files_x):
    image_path = os.path.join(Train_path_x, filename)
    image = imread(image_path)
    image = resize(image, (img_height, img_width), mode='constant', preserve_range=True)
    image = image.astype(np.float32) / 255.0
    new_image = np.stack((image,) * 3, axis=-1)
    X_train[n] = new_image

# Load Y_train images
for n, filename in enumerate(train_files_y):
    mask_path = os.path.join(Train_path_y, filename)
    mask = imread(mask_path)
    if mask.ndim == 3:
        mask = rgb2gray(mask)
    mask = np.expand_dims(resize(mask, (img_height, img_width), mode='constant', preserve_range=True), axis=-1)
    mask = mask.astype(np.float32)
    Y_train[n] = mask

print("X_train shape is:", X_train.shape)
print("Y_train shape is:", Y_train.shape)

def display_image_and_mask(image, mask):
    fig, ax = plt.subplots(1, 2, figsize=(12, 6))
    ax[0].imshow(image)
    ax[0].set_title('Image')
    ax[1].imshow(mask, cmap='gray')
    ax[1].set_title('Mask')
    plt.show()

# Display some samples
for i in range(5):
    display_image_and_mask(X_train[i], Y_train[i])

# Convert masks to binary
Y_train_binary = (Y_train > 0).astype(np.float32)
print("Unique values in the masks:", np.unique(Y_train_binary))


In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_val, y_train, y_val = train_test_split(X_train, Y_train_binary, test_size=0.2, random_state=42)

In [ ]:
x_train = preprocess_input(x_train)
x_val = preprocess_input(x_val)


In [ ]:
model = sm.Unet(BACKBONE, encoder_weights='imagenet')
model.compile(optimizer='adam', loss= sm.losses.bce_jaccard_loss, metrics=[sm.metrics.iou_score])

print(model.summary())

In [ ]:
history = model.fit(
    x_train,
    y_train,
    batch_size=16,
    epochs=20,
    validation_data=(x_val, y_val),
    verbose=1)